In [2]:
import os
import json
import pandas as pd
import traceback

In [1]:
from langchain_openai import ChatOpenAI

In [3]:
from dotenv import load_dotenv

load_dotenv() #take environment variables from .env. 

True

In [4]:
KEY=os.getenv("OPENAI_API_KEY")



In [5]:
llm=ChatOpenAI(api_key=KEY,model="gpt-3.5-turbo",temperature=0.5)

In [6]:
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x777798aab970>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x777793d71240>, root_client=<openai.OpenAI object at 0x777798aab9d0>, root_async_client=<openai.AsyncOpenAI object at 0x777793d71300>, temperature=0.5, model_kwargs={}, openai_api_key=Se

In [10]:
!pip install langchain-classic

  Using cached langchain_classic-1.0.7-py3-none-any.whl.metadata (5.1 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl.metadata (4.2 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached sqlalchemy-2.0.50-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (9.5 kB)
  Using cached greenlet-3.5.1-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (3.8 kB)
Using cached langchain_classic-1.0.7-py3-none-any.whl (1.0 MB)
Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
Using cached sqlalchemy-2.0.50-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (3.2 MB)
Using cached greenlet-3.5.1-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (612 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [langchain-classic]32m4/5 [langchain-classic]


In [12]:
!pip install -U langchain langchain-openai langchain-community langchain-classic

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached aiohttp-3.13.5-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached pydantic_settings-2.14.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached aiohappyeyeballs-2.6.2-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp310-cp310-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (20 kB)
  Using cached multidict-6.7.1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (5.3 kB)
  Using cached propcache-0.5.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached yarl-1.24.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (94 kB)
Using

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_classic.chains import SimpleSequentialChain
from langchain_community.callbacks.manager import get_openai_callback
import PyPDF2




In [16]:
RESPONSE_JSON = {
    "1": {
        "mcq":"multiple choice question",
        "options" :{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",
            "d":"choice here",

        },
        "correct":"correct answer",
    },
    "2": {
        "mcq":"multiple choice question",
        "options" :{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",
            "d":"choice here",

        },
        "correct":"correct answer",
    },
    "3": {
        "mcq":"multiple choice question",
        "options" :{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",
            "d":"choice here",

        },
        "correct":"correct answer",
    },
}

In [17]:
TEMPLATE="""
Text: {text}
you are an expert MCQ maker.Given the above text,it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [18]:
quiz_generation_prompt=PromptTemplate(
    input_variables=["text","number","subject","tone","respose_json"],
    template=TEMPLATE
)

In [20]:
quiz_chain=LLMChain(llm=llm,prompt=quiz_generation_prompt,output_key="quiz",verbose=True)

In [21]:
TEMPLATE="""
You are an expert english gramarian and writer.Given a Multiple Choice Quiz for {subject} students. \
You need to evaluate the complexity of the question and give a complete analysis of the quiz.Only use at max 50 words for complexity
If the quiz is not at per with the cognitve and analytical abilities of the students, \
update the quiz questions which needs to be changed and change the tone that is perfectly fits the student ability
quiz_MCQs:
{quiz}

check from an expert English Writer of the above quiz:
"""

In [22]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject","quiz"],template=TEMPLATE)

In [23]:
review_chain=LLMChain(llm=llm,prompt=quiz_evaluation_prompt,output_key="review",verbose=True)

In [25]:
from langchain_classic.chains import SequentialChain

generate_evaluate_chain = SequentialChain(
    chains=[quiz_chain, review_chain],
    input_variables=[
        "text",
        "number",
        "subject",
        "tone",
        "response_json"
    ],
    output_variables=[
        "quiz",
        "review"
    ],
    verbose=True
)

In [29]:
file_path="/home/gajulas/mcqgen/data.txt"


In [30]:
file_path

'/home/gajulas/mcqgen/data.txt'

In [31]:
with open(file_path,'r') as file:
    TEXT =file.read()
    

In [33]:
print(TEXT)

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.[1] Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.

Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.[3][4]

From a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empirical risk minimisation under this framework.


In [34]:
# serialize the python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON) 

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [35]:
NUMBER=5
SUBJECT="machine learning"
TONE="simple"


In [46]:
#https://python.langchain.com/docs/modules/model_io/llms/token_usage_tracking

#how to setup token usage tracking in langchain
with get_openai_callback() as cb:
    response=generate_evaluate_chain.invoke(
       {
            "text": TEXT,
            "number": NUMBER,
            "subject": SUBJECT,
            "tone": TONE,
            "response_json":json.dumps(RESPONSE_JSON)
       }
       )
   



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Text: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.[1] Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.

Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning.[3][4]

From a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empirical

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [47]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost}")

Total Tokens:0
Prompt Tokens:0
Completion Tokens:0
Total Cost:0.0


In [48]:
response

NameError: name 'response' is not defined

In [50]:
quiz=response.get("quiz")

NameError: name 'response' is not defined

In [52]:
quiz=json.loads(quiz)

NameError: name 'quiz' is not defined

In [54]:
quiz_table_data =[]
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}:{option_value}"
            for option,option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ":mcq,"Choices":options,"Correct":correct})

NameError: name 'quiz' is not defined

In [55]:
quiz_table_data

[]

In [57]:
quiz=pd.DataFrame(quiz_table_data)

In [58]:
quiz.to_csv("machinelearning.csv",index=False)